In [2]:
#Import libraries
import re
import io
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from shapely import wkt
from tqdm import tqdm
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [3]:
#Read datasets
#shape: (479970, 9)
df_construction = pd.read_csv("../../data/street_closures_construction.csv")
print(f'Total rows: {df_construction.shape}')


Total rows: (479970, 9)


In [4]:
#Formatting the columns
def to_snake_case(column_name: str) -> str:
    """Convert a column name to snake_case (e.g. "OnStreetName" -> "on_street_name")."""
    column_name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", column_name)
    column_name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", column_name)
    return (
        column_name.strip()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .lower()
    )
   
#Convert all columns to snake case
df_construction.columns = [to_snake_case(col) for col in df_construction.columns]

In [5]:
#Remove data before 2025 and incorrect data
print(f'Total rows: {df_construction.shape}')

# Convert format of dates
df_construction['work_start_date'] = pd.to_datetime(
    df_construction['work_start_date'], format='%Y %b %d %I:%M:%S %p'
)
df_construction['work_end_date'] = pd.to_datetime(
    df_construction['work_end_date'], format='%Y %b %d %I:%M:%S %p'
)

# Remove data before 2025 since analysis is about after 2025
# shape: (479970, 9) -> (50843, 9)
cutoff_date = pd.Timestamp('2025-01-01 00:00:00')
df_construction = df_construction[df_construction['work_start_date'] >= cutoff_date]
print(f'After date cutoff: {df_construction.shape}')

# Remove end date before start date because this data is invalid
# shape: (50843, 9) -> (50843, 9), no effect
df_construction = df_construction[df_construction['work_end_date'] > df_construction['work_start_date']]
print(f'After date filter: {df_construction.shape}')

# Drop duplicates from OFT codes with same start and work dates
# shape: (50843, 9) -> (35966, 9)
df_construction = df_construction.drop_duplicates(subset=["oft_code", "work_start_date", "work_end_date"])
print(f'After duplicate filter: {df_construction.shape}')

# Remove invalid boroughs
# shape: (35966, 9)-> (35966, 9), no effect
valid_boroughs = ['MANHATTAN', 'BROOKLYN', 'QUEENS', 'BRONX', 'STATEN ISLAND']
df_construction = df_construction[df_construction['borough_name'].str.upper().isin(valid_boroughs)]
print(f'After borough filter: {df_construction.shape}')

Total rows: (479970, 9)
After date cutoff: (50843, 9)
After date filter: (50843, 9)
After duplicate filter: (35966, 9)
After borough filter: (35966, 9)


In [6]:
"""
Impute location_ids

Builds a `location_id` Series (indexed like your closures df) using four
tiers, in order:

  1. WKT spatial join       -- closure geometry already on the row
  2. Census batch geocoder  -- geocode the street/intersection text
  3. Street/borough mode    -- copy the most common location_id seen
                                on the same street+borough elsewhere
  4. Borough mode           -- last resort: most common location_id
                                anywhere in that borough
"""

def build_census_query(row: pd.Series) -> str:
    """e.g. "1 AVENUE and E 14 STREET" """
    def clean(v):
        v = str(v).strip()
        return "" if v.lower() == "nan" else v

    on_st = clean(row.get("on_street_name", ""))
    cross_st = clean(row.get("from_street_name", "")) or clean(row.get("to_street_name", ""))
    if on_st and cross_st:
        return f"{on_st} and {cross_st}"
    return on_st


def get_wkt_location_ids(df: pd.DataFrame, sf: gpd.GeoDataFrame) -> pd.Series:
    """Tier 1: spatial-join rows that have a WKT geometry to a taxi zone.
    Returns a location_id Series indexed like df (NaN where no WKT or no match)."""
    sub = df[df["wkt"].notna()].copy()
    sub["geometry"] = sub["wkt"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(sub, geometry="geometry", crs="EPSG:2263")
    sf_proj = sf.to_crs(gdf.crs)

    # Only geometry goes into the join, so there's no location_id collision to worry about.
    joined = gpd.sjoin(gdf[["geometry"]], sf_proj[["location_id", "geometry"]], how="left", predicate="intersects")
    joined = joined[~joined.index.duplicated(keep="first")]  # keep first match if closure spans 2 zones

    return joined["location_id"].reindex(df.index)


def get_census_location_ids(df: pd.DataFrame, sf: gpd.GeoDataFrame, batch_size: int = 1000) -> pd.Series:
    """Tier 2: geocode street text via the free Census batch API, then spatial-join
    the resulting points to a taxi zone. Returns a location_id Series indexed like df."""
    query = df.apply(build_census_query, axis=1)
    addr_list = query.drop_duplicates().tolist()
    print(f"   Geocoding {len(addr_list):,} unique addresses via Census...")

    matches = []
    for i in tqdm(range(0, len(addr_list), batch_size), desc="   Census batches"):
        chunk = addr_list[i:i + batch_size]
        batch = pd.DataFrame({"id": range(len(chunk)), "street": chunk, "city": "New York", "state": "NY", "zip": ""})
        buf = io.StringIO()
        batch.to_csv(buf, index=False, header=False)

        try:
            resp = requests.post(
                "https://geocoding.geo.census.gov/geocoder/locations/addressbatch",
                files={"addressFile": ("addresses.csv", buf.getvalue(), "text/csv")},
                data={"benchmark": "Public_AR_Current"}, timeout=120,
            )
            result = pd.read_csv(io.StringIO(resp.text), header=None,
                                  names=["id", "input", "status", "type", "matched", "coords", "tiger", "side"])
            result = result[result["status"] == "Match"].copy()
            result[["lon", "lat"]] = result["coords"].str.split(",", expand=True).astype(float)
            result["street"] = result["id"].map(dict(enumerate(chunk)))
            matches.append(result[["street", "lon", "lat"]])
        except Exception as e:
            print(f"   [Warning] batch failed: {e}")

    if not matches:
        return pd.Series(np.nan, index=df.index, name="location_id")

    # street text -> coords lookup, then broadcast back to every row with that text
    coords = pd.concat(matches, ignore_index=True).drop_duplicates(subset="street").set_index("street")
    points_df = pd.DataFrame({"lon": query.map(coords["lon"]), "lat": query.map(coords["lat"])}).dropna()

    points = gpd.GeoDataFrame(
        points_df, geometry=gpd.points_from_xy(points_df["lon"], points_df["lat"]), crs="EPSG:4326"
    ).to_crs(sf.crs)
    joined = gpd.sjoin(points[["geometry"]], sf[["location_id", "geometry"]], how="left", predicate="within")
    joined = joined[~joined.index.duplicated(keep="first")]

    return joined["location_id"].reindex(df.index)


def get_mode_location_ids(df: pd.DataFrame, location_ids: pd.Series, group_cols: list) -> pd.Series:
    """Fill still-missing location_ids with the most common value seen for the
    same group_cols (e.g. street+borough, or just borough) among rows that
    ARE already resolved in location_ids. Returns a Series indexed like df."""
    resolved_mask = location_ids.notna()
    resolved = df.loc[resolved_mask, group_cols].copy()
    resolved["location_id"] = location_ids[resolved_mask]

    mode_lookup = (
        resolved.groupby(group_cols)["location_id"]
        .agg(lambda s: s.value_counts().idxmax())
        .rename("location_id")
    )

    filled = df[group_cols].merge(mode_lookup, on=group_cols, how="left")["location_id"]
    filled.index = df.index
    return filled


def impute_close_location_ids(df: pd.DataFrame, sf: gpd.GeoDataFrame) -> pd.Series:
    """Returns a location_id Series indexed like df_close -- join it onto your
    dataset however you'd like, e.g. df_close["location_id"] = imputed_ids."""

    location_ids = get_wkt_location_ids(df, sf)
    print(f"Tier 1 (WKT):          {location_ids.notna().sum():,} resolved, {location_ids.isna().sum():,} remaining")

    still_missing = df[location_ids.isna()]
    census_ids = get_census_location_ids(still_missing, sf) if not still_missing.empty else pd.Series(dtype=object)
    location_ids = location_ids.fillna(census_ids)
    print(f"Tier 2 (Census):       {location_ids.notna().sum():,} resolved total, {location_ids.isna().sum():,} remaining")

    street_ids = get_mode_location_ids(df, location_ids, group_cols=["on_street_name", "borough_name"])
    location_ids = location_ids.fillna(street_ids)
    print(f"Tier 3 (street mode):  {location_ids.notna().sum():,} resolved total, {location_ids.isna().sum():,} remaining")

    borough_ids = get_mode_location_ids(df, location_ids, group_cols=["borough_name"])
    location_ids = location_ids.fillna(borough_ids)
    print(f"Tier 4 (borough mode): {location_ids.notna().sum():,} resolved total, {location_ids.isna().sum():,} unresolved")

    return location_ids.rename("location_id")


if __name__ == "__main__":
    #read taxi zones
    sf = gpd.read_file("../../data/taxi_zones/taxi_zones.shp")
    #format data to snake case
    sf.columns = [to_snake_case(col) for col in sf.columns]

    #impute location ids
    location_ids = impute_close_location_ids(df_construction, sf)
    
    #add location ids
    #shape: (35966, 9) -> (35966, 10)
    df_construction["location_id"] = location_ids
    print(f'After imputation: {df_construction.shape}')
    
    

Tier 1 (WKT):          35,934 resolved, 32 remaining
   Geocoding 22 unique addresses via Census...


   Census batches: 100%|██████████| 1/1 [00:01<00:00,  1.81s/it]


Tier 2 (Census):       35,954 resolved total, 12 remaining
Tier 3 (street mode):  35,965 resolved total, 1 remaining
Tier 4 (borough mode): 35,966 resolved total, 0 unresolved
After imputation: (35966, 10)


In [7]:
# Construction closures
print("construction")
# Rename columns
df_construction = df_construction.rename(columns={
    'work_start_date': 'start_time',
    'work_end_date': 'end_time',
    'borough_name': 'borough'
})
# shape: (35966, 10) -> (35966, 4)
df_construction = df_construction[["start_time", "end_time", "borough", "location_id"]]
print(f"After selectioin: {df_construction.shape}")

construction
After selectioin: (35966, 4)


In [ ]:
#save file
df_construction.to_csv("../../data/processed/construction_closures.csv", index=False)